# Pharmakaksha · BP701T · Unit V · Research Methodology and Design of Experiments
**Learn · Grow · Achieve**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajreddys/learn-pharmakaksha-site/blob/main/content/BP701T%20Biostatistics%20and%20Research%20Methodology%20%28Theory%29/BP701T_Unit5_Research_Methodology_and_DoE.ipynb) &nbsp; [Open on learn.pharmakaksha.com](https://learn.pharmakaksha.com/notebooks/index.html?path=BP701T%20Biostatistics%20and%20Research%20Methodology%20%28Theory%29/BP701T_Unit5_Research_Methodology_and_DoE.ipynb)

Companion notebook for BP701T Unit V study notes, slides and video. Every worked example from the unit is here, in order, followed by lab exercises and solutions.

> **How to use this notebook:** BP701T is a hand-calculation course. Solve each example with your calculator first, using the study notes, then run the cell to check your answer. In the exam you will not have Python.

## Using learn.pharmakaksha.com
- **First run is slow:** Python loads in your browser, which takes about 10 seconds the first time.
- **Kernel indicator:** the circle at the top right is filled while a cell runs and empty when it is idle.
- **Restart:** Kernel → Restart Kernel if something gets stuck, then run the cells again from the top.
- **Saving:** your changes are saved only in this browser. Download the notebook (File → Download) to keep a copy.
- **Start over:** delete your copy in the file browser to get a fresh notebook from the site.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
print("Ready")

## 5.2 Sample size
**(a) Estimating a prevalence.** Expected prevalence 20%, precision ±5%, 95% confidence: n = Z² p(1 − p) ÷ d² = 245.9 → 246.
**(b) Comparing two proportions.** ADR rate 30% vs 15%, α = 0.05 two-sided, 80% power → about 121 per group.

In [ ]:
z = stats.norm.ppf(0.975)
n_prev = z**2 * 0.20 * 0.80 / 0.05**2
print(f"(a) n = {n_prev:.1f} -> {int(np.ceil(n_prev))}")
p1, p2 = 0.30, 0.15; pbar = (p1 + p2) / 2; zb = stats.norm.ppf(0.80)
n2 = (z*np.sqrt(2*pbar*(1-pbar)) + zb*np.sqrt(p1*(1-p1) + p2*(1-p2)))**2 / (p1 - p2)**2
print(f"(b) n per group = {n2:.1f} -> {int(np.ceil(n2))}")
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
print("statsmodels (arcsine method):", round(NormalIndPower().solve_power(proportion_effectsize(p1, p2), alpha=0.05, power=0.8), 1))

### Sampling and randomisation
Simple random sample of 10 patients from a register of 200, and a block-randomised allocation list (blocks of 4) for 12 patients.

In [ ]:
rng = np.random.default_rng(2026)
print("Random sample of IDs:", sorted(int(i) for i in rng.choice(np.arange(1, 201), size=10, replace=False)))
alloc = np.concatenate([rng.permutation(["Drug", "Drug", "Placebo", "Placebo"]) for _ in range(3)])
print(pd.DataFrame({"patient": range(1, 13), "arm": alloc}).to_string(index=False))

## 5.3 Observational designs: relative risk and odds ratio
**Cohort:** 30 of 200 exposed and 10 of 200 unexposed patients develop the outcome → RR = 0.15 ÷ 0.05 = 3.0.
**Case–control:** 40 of 100 cases and 20 of 100 controls used NSAIDs → OR = (40 × 80) ÷ (60 × 20) = 2.67.

In [ ]:
a, b, c, d = 30, 170, 10, 190
print("RR =", round((a/(a+b)) / (c/(c+d)), 2))
a, b, c, d = 40, 60, 20, 80   # cases exposed/unexposed, controls exposed/unexposed
OR = (a*d) / (b*c); se = np.sqrt(1/a + 1/b + 1/c + 1/d)
print(f"OR = {OR:.2f}, 95% CI {np.exp(np.log(OR) - 1.96*se):.2f} to {np.exp(np.log(OR) + 1.96*se):.2f}")

## 5.4 A decision tree for choosing the test
The function follows the table in the notes: outcome type → number of groups → paired or independent.

In [ ]:
def choose_test(outcome, groups=2, paired=False, normal=True):
    if outcome == "association":
        return "Pearson r (normal) / Spearman rho (ranks); regression to predict"
    if outcome == "categorical":
        return "McNemar test" if paired else "Chi-square test (Fisher exact if expected < 5)"
    parametric = outcome == "continuous" and normal
    if groups == 1:
        return "One-sample t / Z test" if parametric else "Wilcoxon signed-rank (one sample)"
    if groups == 2:
        if paired: return "Paired t-test" if parametric else "Wilcoxon signed-rank test"
        return "Unpaired t-test" if parametric else "Mann-Whitney U test"
    if paired: return "Two-way ANOVA / repeated measures" if parametric else "Friedman test"
    return "One-way ANOVA" if parametric else "Kruskal-Wallis test"

cases = [("BP before vs after, 10 patients", dict(outcome="continuous", paired=True)),
         ("Pain score (0-10), drug vs placebo", dict(outcome="ordinal")),
         ("Dissolution of 3 brands", dict(outcome="continuous", groups=3)),
         ("ADR yes/no, drug vs placebo", dict(outcome="categorical")),
         ("Skewed LOS, 3 wards", dict(outcome="continuous", groups=3, normal=False)),
         ("Dose vs response", dict(outcome="association"))]
for label, kw in cases: print(f"{label:38s} -> {choose_test(**kw)}")

## 5.6 Design of experiments: a 2² factorial
Factor A = binder (PVP 2% / 4%), factor B = disintegrant (crospovidone 2% / 5%); response = % dissolved at 30 min.
Runs: (1) = 70, a = 60, b = 85, ab = 80. Hand answers: effect A = −7.5, effect B = +17.5, interaction AB = +2.5; model y = 73.75 − 3.75A + 8.75B + 1.25AB.

In [ ]:
runs = pd.DataFrame({"run": ["(1)", "a", "b", "ab"], "A": [-1, 1, -1, 1], "B": [-1, -1, 1, 1], "y": [70, 60, 85, 80]})
runs["AB"] = runs.A * runs.B
for col in ["A", "B", "AB"]:
    eff = runs.y[runs[col] == 1].mean() - runs.y[runs[col] == -1].mean()
    print(f"effect {col:2s} = {eff:+.1f}   coefficient = {eff/2:+.2f}")
from statsmodels.formula.api import ols
print(ols("y ~ A * B", runs).fit().params.round(2))
for lvl, ls in [(-1, "o--"), (1, "s-")]:
    sub = runs[runs.B == lvl]; plt.plot(sub.A, sub.y, ls, label=f"disintegrant {'low' if lvl < 0 else 'high'}")
plt.xticks([-1, 1], ["binder low", "binder high"]); plt.ylabel("% dissolved at 30 min")
plt.title("Nearly parallel lines = small interaction"); plt.legend(); plt.show()

### How many runs does each design need?
Full factorial 2^k; Plackett–Burman screens up to N − 1 factors in N runs (N a multiple of 4); central composite = 2^k + 2k + centre points.

In [ ]:
import itertools
ks = [3, 5, 7, 11]
print(pd.DataFrame({"factors k": ks,
                    "full factorial 2^k": [2**k for k in ks],
                    "CCD (6 centre pts)": [2**k + 2*k + 6 for k in ks],
                    "Plackett-Burman": [max(8, 4 * int(np.ceil((k + 1) / 4))) for k in ks]}).to_string(index=False))
full = pd.DataFrame(list(itertools.product([-1, 1], repeat=3)), columns=["A", "B", "C"])
print("\n2^3 design matrix:\n", full.to_string(index=False))
alpha = 2**(3/4)
axial = [[s*alpha if i == j else 0 for j in range(3)] for i in range(3) for s in (-1, 1)]
print(f"\nCCD axial points (alpha = {alpha:.3f}):\n", pd.DataFrame(axial, columns=["A", "B", "C"]).round(3).to_string(index=False))

## Lab exercises
1. A survey will estimate the proportion of pharmacies that dispense antibiotics without prescription; expected 40%, precision ±5%. Find n.
2. In a case–control study, 45 of 150 cases and 25 of 150 controls took a drug. Find the OR.
3. Choose the test: (a) haemoglobin before and after iron in 15 patients; (b) satisfaction (5-point scale) across 4 pharmacies; (c) cure yes/no in two antibiotic groups.
4. A 2² factorial on hardness gave (1) = 6, a = 9, b = 5, ab = 10. Find the main effects and the interaction.

### Solutions

In [ ]:
print("1: n =", int(np.ceil(stats.norm.ppf(0.975)**2 * 0.4 * 0.6 / 0.05**2)))
print("2: OR =", round((45 * 125) / (105 * 25), 2))
print("3a:", choose_test("continuous", paired=True)); print("3b:", choose_test("ordinal", groups=4)); print("3c:", choose_test("categorical"))
y = {"(1)": 6, "a": 9, "b": 5, "ab": 10}
print("4: A =", ((y["a"] + y["ab"]) - (y["(1)"] + y["b"])) / 2, "| B =", ((y["b"] + y["ab"]) - (y["(1)"] + y["a"])) / 2,
      "| AB =", ((y["(1)"] + y["ab"]) - (y["a"] + y["b"])) / 2)